# 07 · Sesión 2 · Práctica final

**Duración:** 1h 30min
**Objetivo:** comparar evaluación simple, validación cruzada y efecto de outliers en un caso de regresión. Si te atascas, mira la **solución orientativa** debajo de cada tarea.

## Dataset

Creamos un dataset de regresión con 3 variables y le añadimos 8 outliers en `y` para que el efecto de la limpieza sea claro.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

In [ ]:
X, y = make_regression(
    n_samples=180, n_features=3, n_informative=3, noise=20, random_state=12
)

df = pd.DataFrame(X, columns=['x1', 'x2', 'x3'])
df['y'] = y

# Añadimos outliers
df.loc[:7, 'y'] = df.loc[:7, 'y'] * 4 + 250
df.head()

---

### Tarea 1 · Exploración

Usa `shape`, `describe()` y un histograma de `y` para entender el rango y la distribución.

In [ ]:
# Tu código aquí

**Solución orientativa**

In [ ]:
print('Shape:', df.shape)
print()
df.describe().round(2)

plt.figure(figsize=(7, 4))
plt.hist(df['y'], bins=25, color='steelblue', edgecolor='k')
plt.xlabel('y')
plt.ylabel('frecuencia')
plt.title('Distribución de y (con outliers)')
plt.show()

---

### Tarea 2 · Train / test split + entrenar

- `test_size=0.2`, `random_state=42`.
- Entrena un `LinearRegression`.
- Calcula y guarda MAE y R² sobre el test.

In [ ]:
# Tu código aquí

**Solución orientativa**

In [ ]:
X_full = df.drop(columns='y')
y_full = df['y']

X_train, X_test, y_train, y_test = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42
)

model = LinearRegression().fit(X_train, y_train)
pred  = model.predict(X_test)

mae_inicial = mean_absolute_error(y_test, pred)
r2_inicial  = r2_score(y_test, pred)

print(f'MAE : {mae_inicial:.2f}')
print(f'R²  : {r2_inicial:.3f}')

---

### Tarea 3 · Validación cruzada inicial

Aplica `cross_val_score(cv=5, scoring='r2')` sobre el dataset completo (sin limpiar) y guarda la media. **No reentrenes el modelo**: pásalo directo a `cross_val_score`.

In [ ]:
# Tu código aquí

**Solución orientativa**

In [ ]:
cv_inicial = cross_val_score(model, X_full, y_full, cv=5, scoring='r2').mean()
print(f'CV R² (con outliers): {cv_inicial:.3f}')

---

### Tarea 4 · Detectar outliers con IQR sobre `y`

- Calcula Q1, Q3 e IQR.
- Define `lower = Q1 - 1.5*IQR` y `upper = Q3 + 1.5*IQR`.
- Cuenta cuántos outliers hay.

In [ ]:
# Tu código aquí

**Solución orientativa**

In [ ]:
q1 = df['y'].quantile(0.25)
q3 = df['y'].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

mask = (df['y'] >= lower) & (df['y'] <= upper)
print(f'Límite inferior: {lower:.2f}')
print(f'Límite superior: {upper:.2f}')
print(f'Sin outliers : {mask.sum()} filas')
print(f'Outliers     : {(~mask).sum()} filas')

---

### Tarea 5 · Dataset limpio y nuevo entrenamiento

1. Crea `df_clean` filtrando las filas sin outliers.
2. Vuelve a hacer `train_test_split` y entrenar (con los mismos parámetros).
3. Calcula MAE y R² sobre el test limpio.

In [ ]:
# Tu código aquí

**Solución orientativa**

In [ ]:
df_clean = df[mask].copy()

X_clean = df_clean.drop(columns='y')
y_clean = df_clean['y']

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clean, y_clean, test_size=0.2, random_state=42
)

model_clean = LinearRegression().fit(X_train_c, y_train_c)
pred_clean  = model_clean.predict(X_test_c)

mae_limpio = mean_absolute_error(y_test_c, pred_clean)
r2_limpio  = r2_score(y_test_c, pred_clean)

print(f'MAE : {mae_limpio:.2f}')
print(f'R²  : {r2_limpio:.3f}')

---

### Tarea 6 · Validación cruzada final + tabla comparativa

1. Aplica de nuevo `cross_val_score` sobre el dataset limpio.
2. Mete los 4 números en una tabla: MAE, R² y CV R², antes y después.

In [ ]:
# Tu código aquí

**Solución orientativa**

In [ ]:
cv_limpio = cross_val_score(model_clean, X_clean, y_clean, cv=5, scoring='r2').mean()

resumen = pd.DataFrame({
    'escenario':  ['Con outliers', 'Sin outliers'],
    'MAE':        [round(mae_inicial, 2), round(mae_limpio, 2)],
    'R² (test)':  [round(r2_inicial,  3), round(r2_limpio,  3)],
    'CV R² (media)': [round(cv_inicial, 3), round(cv_limpio, 3)],
})
resumen

---

### Reto opcional · Visualiza el efecto

Dibuja un `boxplot` de `y` antes y después de limpiar outliers. Puedes usar `plt.boxplot([df['y'], df_clean['y']], labels=['con outliers', 'sin outliers'])`.

In [ ]:
# Tu código aquí

---

## Preguntas finales

- ¿Qué métrica cambió más al limpiar los outliers?
- ¿La validación cruzada cuenta la misma historia que el train/test split?
- ¿Borrarías siempre los outliers? ¿En qué casos NO lo harías?